In [1]:
import os
import shutil
from glob import glob

# === Paths ===
src_base = "/home/ubuntu/additional_drive/shwan_data/yolo_training/all_dataset_marge/"
dst_base = "all_dataset_singleclass/"

src_img_train = os.path.join(src_base, "images/train")
src_img_val = os.path.join(src_base, "images/val")
src_lbl_train = os.path.join(src_base, "labels/train")
src_lbl_val = os.path.join(src_base, "labels/val")

dst_img_train = os.path.join(dst_base, "images/train")
dst_img_val = os.path.join(dst_base, "images/val")
dst_lbl_train = os.path.join(dst_base, "labels/train")
dst_lbl_val = os.path.join(dst_base, "labels/val")

# === Create directories ===
for d in [dst_img_train, dst_img_val, dst_lbl_train, dst_lbl_val]:
    os.makedirs(d, exist_ok=True)

print("✅ Destination folders created")

# === Helper function to process label files ===
def process_and_copy(src_img_dir, src_lbl_dir, dst_img_dir, dst_lbl_dir):
    images = glob(os.path.join(src_img_dir, "*"))
    print(f"📦 Processing {len(images)} images from {src_img_dir}")

    for img_path in images:
        img_name = os.path.basename(img_path)
        lbl_name = os.path.splitext(img_name)[0] + ".txt"
        lbl_path = os.path.join(src_lbl_dir, lbl_name)

        # Skip if annotation missing
        if not os.path.exists(lbl_path):
            continue

        # ✅ Rewrite all labels with class "0"
        new_lines = []
        with open(lbl_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                parts[0] = "0"  # Force single class
                new_lines.append(" ".join(parts))

        # Copy image and save new label
        shutil.copy(img_path, os.path.join(dst_img_dir, img_name))
        with open(os.path.join(dst_lbl_dir, lbl_name), "w") as f:
            f.write("\n".join(new_lines))

    print(f"✅ Done: {src_img_dir} → {dst_img_dir}")

# === Run for train and val ===
process_and_copy(src_img_train, src_lbl_train, dst_img_train, dst_lbl_train)
process_and_copy(src_img_val, src_lbl_val, dst_img_val, dst_lbl_val)

# === Create new YAML file ===
yaml_path = os.path.join(dst_base, "data.yaml")
with open(yaml_path, "w") as f:
    f.write("train: images/train\n")
    f.write("val: images/val\n")
    f.write("nc: 1\n")
    f.write("names: ['object']\n")

print("\n✅ ALL DONE — Single-class dataset created successfully!")
print(f"✅ New dataset path: {dst_base}")
print(f"✅ YAML saved to: {yaml_path}")


✅ Destination folders created
📦 Processing 2709 images from /home/ubuntu/additional_drive/shwan_data/yolo_training/all_dataset_marge/images/train
✅ Done: /home/ubuntu/additional_drive/shwan_data/yolo_training/all_dataset_marge/images/train → all_dataset_singleclass/images/train
📦 Processing 183 images from /home/ubuntu/additional_drive/shwan_data/yolo_training/all_dataset_marge/images/val
✅ Done: /home/ubuntu/additional_drive/shwan_data/yolo_training/all_dataset_marge/images/val → all_dataset_singleclass/images/val

✅ ALL DONE — Single-class dataset created successfully!
✅ New dataset path: all_dataset_singleclass/
✅ YAML saved to: all_dataset_singleclass/data.yaml
